# Building a RAG system with LangChain and FAISS

# FIASS

FIASS is a library for efficient similar search and clustring of dense vector

Key advantages:

1. Extremely fast similarity search
2. Memory efficeht
3. Supports GPU acceleration
4. Can handle millions of vectors

How it works:
- Indexes vectors for fast nearest neighbour search
- Returns most similar vectors based on distance metics

In [1]:
# Load libraries

import os
from dotenv import load_dotenv
import numpy as np
from typing import List, Dict, Any, Optional
import warnings

warnings.filterwarnings('ignore')

# Langchain core imports
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import (RunnablePassthrough)

from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# Langchain specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyMuPDFLoader, DirectoryLoader
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


load_dotenv(override=True)

True

In [2]:
dir_loader = DirectoryLoader(
    "../data/text_files",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

documents = dir_loader.load()

print(f"Loaded {len(documents)} documents")
documents

Loaded 3 documents


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation,[38] an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.\n\nGuido van Rossum began working on Python in the late 1980s as a successor to the ABC programming language. Python 3.0, released in 2008, was a major revision and not completely backward-compatible with earlier versions. Beginning with Python 3.5,[39] capabilities and keywords for typing were added to the language, allowing optional static typing.[40] As of 2026, the Python Software Foundation supports Python 3.10, 3.11, 3.12, 3.13, and 3.14, following the project\'s annual release cycle and five-year support policy. Python 3.15.0rc1 (whic

In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    separators=["\n\n", "\n", " ", ""],
    length_function = len
)

## Split the documents into chunks
chunks = text_splitter.split_documents(documents)
print(chunks)
print(f"Length of chunks: {len(chunks)}")
for chunk in chunks:
    print(chunk)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation,[38] an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.'), Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content="Guido van Rossum began working on Python in the late 1980s as a successor to the ABC programming language. Python 3.0, released in 2008, was a major revision and not completely backward-compatible with earlier versions. Beginning with Python 3.5,[39] capabilities and keywords for typing were added to the language, allowing optional static typing.[40] As of 2026, the Python Software Foundation supports Python 3.10, 3.11, 3.12, 3.13, and 3.14, following the p

In [4]:
### Load the embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name = os.getenv("HUGGING_FACE_EMBEDDING_MODEL")
)

embedding_model

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 16781.13it/s]


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [5]:
## Example create embedding for simple text
sample_text = "What is machine learning?"
sample_embedding = embedding_model.embed_query(sample_text)

sample_embedding

[-0.01995455101132393,
 0.00987799558788538,
 0.010249645449221134,
 0.029553702101111412,
 0.027186434715986252,
 -0.01929652877151966,
 -0.024129629135131836,
 -0.037735715508461,
 -0.04105418920516968,
 -0.0014749818947166204,
 -0.07607676088809967,
 0.03687207028269768,
 0.053723834455013275,
 -0.07053521275520325,
 -0.08609341084957123,
 0.021006831899285316,
 -0.04226335883140564,
 0.028448930010199547,
 -0.03837532550096512,
 -0.0659191757440567,
 0.013949614949524403,
 0.012275371700525284,
 -0.06821972876787186,
 0.028625069186091423,
 0.014761178754270077,
 0.03223147615790367,
 0.008694306947290897,
 0.01306209247559309,
 -0.015726685523986816,
 0.01889656111598015,
 0.020023880526423454,
 -0.03929709643125534,
 0.01564716175198555,
 0.03339802473783493,
 -0.07339107990264893,
 0.05935781076550484,
 -0.012385821901261806,
 0.024584617465734482,
 0.020107092335820198,
 -0.020254511386156082,
 -0.050247691571712494,
 -0.09044113755226135,
 -0.015374395996332169,
 -0.0036124212

In [7]:
# Batch embedding
texts = ["AI", "cat", "dog", "mango", "orange"]
batch_embeddings = embedding_model.embed_documents(texts)

for embedding in batch_embeddings:
    print(embedding)

[-0.036539267748594284, -0.01516442559659481, 0.016432465985417366, 0.010568794794380665, 0.006010566372424364, -0.018473224714398384, 0.08546526730060577, 0.020968424156308174, 0.027815379202365875, 0.012431630864739418, -0.02937648445367813, -0.03113524056971073, 0.03491253778338432, -0.0181508157402277, -0.0649847611784935, 0.05168252810835838, -0.019606158137321472, -0.015734143555164337, -0.13371671736240387, -0.09645996242761612, -0.025471754372119904, -0.0014895610511302948, -0.006349354982376099, -0.025820672512054443, -0.027371767908334732, 0.12268997728824615, -0.007792433723807335, -0.038522735238075256, 0.0143834687769413, -0.09218423813581467, 0.008695755153894424, 0.002613367745652795, 0.0910346582531929, -0.030313553288578987, -0.09604638069868088, 0.02228911593556404, -0.09024310857057571, -0.032947346568107605, 0.0715833455324173, -0.008893120102584362, -0.025708958506584167, -0.07913962751626968, 0.014530383050441742, -0.07420430332422256, 0.08045011013746262, 0.07804

## Create FAISS Vector database

In [8]:
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print(f"Vector store created with {vector_store.index.ntotal} vectors")

Vector store created with 114 vectors


In [9]:
vector_store

In [10]:
## Save vector store for later use
vector_store.save_local("faiss_index")
print("Vector store saved to 'faiss_index' directory")

Vector store saved to 'faiss_index' directory


In [11]:
# Load vector store
loaded_vector_store = FAISS.load_local(
    "faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

print(f"Loaded vector store contains {loaded_vector_store.index.ntotal} vectors")

Loaded vector store contains 114 vectors


In [12]:
query = "What is python?"

results = vector_store.similarity_search(query, k = 3)
print(results)

[Document(id='b45953b3-4e69-4528-a4f0-45b51f5a314e', metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation,[38] an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.'), Document(id='93a2223b-c55c-4a39-beae-57b88bb66f80', metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python is widely taught as an introductory programming language.[42]\n\nHistory\nMain article: History of Python'), Document(id='36eed48c-d9e3-400a-9a58-2f8d7854f8e0', metadata={'source': '../data/text_files/python_intro.txt'}, page_content="Python is a multi-paradigm programming language. Object-oriented programming and structured programming are fully supported, and many

In [14]:
print(f"Query: {query} \n")
print("Top 3 similar chunks:")
for i, doc in enumerate(results):
    print(f"\n{i+1}. Source: {doc.metadata['source']}")
    print(f"   Content: {doc.page_content[:200]}...")

Query: What is python? 

Top 3 similar chunks:

1. Source: ../data/text_files/python_intro.txt
   Content: Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation,[38] an extensive ("batteries-inc...

2. Source: ../data/text_files/python_intro.txt
   Content: Python is widely taught as an introductory programming language.[42]

History
Main article: History of Python...

3. Source: ../data/text_files/python_intro.txt
   Content: Python is a multi-paradigm programming language. Object-oriented programming and structured programming are fully supported, and many of their features support functional programming and aspect-orient...


In [17]:
### Similarity search with score
result_with_score = vector_store.similarity_search_with_score(query, k = 3)

In [18]:
print(f"Query: {query} \n")
print("Top 3 similar chunks:")
for doc, score in result_with_score:
    print(f"\nScore: {score:.3f}")
    print(f"\nSource: {doc.metadata['source']}")
    print(f"   Content: {doc.page_content[:200]}...")

Query: What is python? 

Top 3 similar chunks:

Score: 0.328

Source: ../data/text_files/python_intro.txt
   Content: Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation,[38] an extensive ("batteries-inc...

Score: 0.567

Source: ../data/text_files/python_intro.txt
   Content: Python is widely taught as an introductory programming language.[42]

History
Main article: History of Python...

Score: 0.608

Source: ../data/text_files/python_intro.txt
   Content: Python is a multi-paradigm programming language. Object-oriented programming and structured programming are fully supported, and many of their features support functional programming and aspect-orient...


In [19]:
chunks

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation,[38] an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.'),
 Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content="Guido van Rossum began working on Python in the late 1980s as a successor to the ABC programming language. Python 3.0, released in 2008, was a major revision and not completely backward-compatible with earlier versions. Beginning with Python 3.5,[39] capabilities and keywords for typing were added to the language, allowing optional static typing.[40] As of 2026, the Python Software Foundation supports Python 3.10, 3.11, 3.12, 3.13, and 3.14, following the 

## Build RAG Chain with LCEL

In [20]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(os.getenv("OLLAMA_CHAT_MODEL"))

llm

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.4.0'}}, model='Gemma4')

In [21]:
#1. Simple RAG chain with LCEL
simple_prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:
    
    Context: {context}
    
    Answer:"""
)



In [23]:
retriever = vector_store.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k": 3}
)

retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x123149400>, search_kwargs={'k': 3})

In [25]:
from typing import List

# Format document for the prompt
def format_docs(docs: List[Document]) -> str:
    """Format documents for insertion into prompt"""
    formatted = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get('source', 'unknown')
        formatted.append(f"Document {i+1} (source: {source}): \n{doc.page_content}")
    return "\n\n".join(formatted)

In [38]:
simple_rag_chain = (
    {
        "context": retriever | format_docs, "question":RunnablePassthrough()
    }
    | simple_prompt
    | llm
    | StrOutputParser()
)

simple_rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x123149400>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n\n    Context: {context}\n\n    Answer:'), additional_kwargs={})])
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.4.0'}}, model='Gemma4')
| StrOutputParser()

In [29]:
### Conversational RAG chain
conversational_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful AI assistant. Use the provided context to answer questions."),
        ('placeholder', "{chat_history}"),
        ("human", "Context: {context} \n\n Question: {input}")
    ]
)

In [30]:
def create_conversational_rag():
    """Create a conversational RAG chain with memory"""
    return (
        RunnablePassthrough.assign(
            context = lambda x: format_docs(retriever.invoke(x["input"]))
        )
        | conversational_prompt
        | llm
        | StrOutputParser()
    )

conversational_rag = create_conversational_rag()

In [31]:
conversational_rag

RunnableAssign(mapper={
  context: RunnableLambda(lambda x: format_docs(retriever.invoke(x['input'])))
})
| ChatPromptTemplate(input_variables=['context', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')] | typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')] | typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')] | typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')] | typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')] | typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')] | typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')] | typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')] | typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='Chat

In [32]:
### Streaming RAG Chain
streaming_reg_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | simple_prompt
    | llm
)

In [33]:
streaming_reg_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x123149400>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n\n    Context: {context}\n\n    Answer:'), additional_kwargs={})])
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.4.0'}}, model='Gemma4')

In [42]:
# Test function for different chain type
def test_rag_chains(question: str):
    """Test all RAG chain varients"""
    print(f"Question: {question}")
    print("="*150)

    #1. Simple RAG
    print('\n1. Simple RAG Chain:')
    answer = simple_rag_chain.invoke(question)
    print(f"Answer: {answer}")

    #2. Streaming RAG
    print('\n1. Streaming RAG Chain:')
    answer = streaming_reg_chain.invoke(question)
    print(f"Answer: {answer.content}")


In [43]:
test_rag_chains("What is the difference between AI and machine learning?")

Question: What is the difference between AI and machine learning?

1. Simple RAG Chain:
Answer: Artificial intelligence (AI) is defined as a set of technologies that enables computers to perform advanced tasks, such as understanding language, analyzing data, and providing helpful suggestions—tasks that once required human intelligence. AI is considered a broad, transformational field that has the potential to bring meaningful and positive change to people, societies, and the world.

**How AI Works:**
Fundamentally, AI systems rely on data, algorithms, and computational power. AI improves and learns by being exposed to vast amounts of data, which serves as the training material. By analyzing this data, the system identifies complex patterns and relationships. For AI performance, both the quality and the quantity of this data are crucial.

**A Key Component:**
One key area within AI is Machine Learning (ML). ML is a type of AI where systems are able to learn from data to make predictions